## DSC355 USL Championship 2025 — Model

**Author:** James Sneddon

**Date:** May 10, 2026

**Modified By:** James Sneddon

## Description
Ridge regression and gradient-boosted trees predicting match points (0/1/3) from goalkeeper Goals Added sub-components and match-level team statistics. Goalkeeper season aggregates are minutes-weighted and joined to the match-level feature-engineered dataset; playoff matches are excluded.

In [1]:
import pandas as pd
import numpy as np
import re

# model building, tuning, and evaluation
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

## EDA

In [2]:
# match-level team stats from the feature engineering notebook
match_df = pd.read_csv('data/USL_Championship_2025_Feature_Engineered.csv')
# goalkeeper season totals from American Soccer Analysis
gk_df    = pd.read_csv('data/american_soccer_analysis_uslc_goals-added_goalkeepers_2026-05-09.csv', index_col=0)

print(f'Match rows : {len(match_df):,}')
print(f'GK rows    : {len(gk_df)}')

Match rows : 746
GK rows    : 56


In [3]:
abbrev_map = {
    'BHM': 'Birmingham Legion',     'CHS': 'Charleston Battery',
    'DET': 'Detroit City',           'HFD': 'Hartford Athletic',
    'IND': 'Indy Eleven',            'LEX': 'Lexington SC',
    'LDN': 'Loudoun United',         'LOU': 'Louisville City',
    'MIA': 'Miami',                  'NC':  'North Carolina FC',
    'PIT': 'Pittsburgh Riverhounds', 'RI':  'Rhode Island',
    'TBR': 'Tampa Bay Rowdies',      'COS': 'Colorado Springs',
    'ELP': 'El Paso Locomotive',     'TUL': 'FC Tulsa',
    'LV':  'Las Vegas Lights',       'MB':  'Monterey Bay FC',
    'NM':  'New Mexico United',      'OAK': 'Oakland Roots',
    'OC':  'Orange County SC',       'PHX': 'Phoenix Rising',
    'SAC': 'Sacramento Republic',    'SA':  'San Antonio',
}

gk_cols = ['Claiming', 'Fielding', 'Handling', 'Passing', 'Shotstopping', 'Sweeping', 'Goals Added']

# exclude players who split the season across two clubs
gk_single = gk_df[~gk_df['Team'].str.contains(',')].copy()
gk_single['Team_Full'] = gk_single['Team'].map(abbrev_map)

# weight each goalkeeper's stats by minutes so starters count more than backups
def weighted_avg(group):
    w = group['Minutes']
    out = {f'GK_{c.replace(" ", "_")}': np.average(group[c], weights=w) for c in gk_cols}
    out['GK_Minutes'] = w.sum()
    return pd.Series(out)

gk_team = (
    gk_single
    .groupby('Team_Full', group_keys=False)
    .apply(weighted_avg)
    .reset_index()
)

# one GK row per team, then broadcast to every match that team played
df = match_df.merge(gk_team, left_on='Team', right_on='Team_Full', how='inner')
print(f'Teams with GK data : {len(gk_team)}')
print(f'Rows after join    : {len(df):,}  |  Teams : {df["Team"].nunique()}')

Teams with GK data : 24
Rows after join    : 746  |  Teams : 24


Two players (Bill Hamid, Nicolas Campisi) split the 2025 season across MIA and TBR; excluding them leaves Miami and Tampa Bay with lower minute coverage but avoids splitting fractional stats across clubs.

## Preprocessing and Feature Extraction

In [4]:
# derive home/away status and points earned from the score string in the Match column
def get_features(row):
    m = re.search(r'(\d+):(\d+)', row['Match'])
    hs, as_ = int(m.group(1)), int(m.group(2))
    home = re.split(r'\s*\(?\w\)?\s*\d+:\d+', row['Match'])[0].split(' - ')[0].strip()
    is_home = row['Team'] == home
    ms, os_ = (hs, as_) if is_home else (as_, hs)
    r = 'W' if ms > os_ else ('L' if ms < os_ else 'D')
    return pd.Series(
        [int(is_home), r, 3 if r == 'W' else 1 if r == 'D' else 0],
        index=['Is_Home', 'Result', 'Points']
    )

if 'Points' not in df.columns:
    df = pd.concat([df, df.apply(get_features, axis=1)], axis=1)

# knockout format and extra time make playoff matches incomparable to the regular season
df = df[df['Season_Phase'] != 'Playoffs'].copy()

print(f'Rows after playoff exclusion : {len(df):,}')
print('\nPoints distribution:')
print(df['Points'].value_counts().sort_index())

Rows after playoff exclusion : 716

Points distribution:
Points
0    263
1    190
3    263
Name: count, dtype: int64


In [5]:
# GK_Goals_Added is a composite of the six sub-components — Ridge handles the resulting collinearity
numeric_features = [
    'GK_Shotstopping', 'GK_Claiming', 'GK_Sweeping',
    'GK_Handling', 'GK_Fielding', 'GK_Passing', 'GK_Goals_Added',
    'xG', 'Shots on target', 'Possession %', 'PPDA',
    'xG_Diff', 'Shot_Conversion_Rate',
    'Shots against on target', 'Deep completed passes',
    'Touches in penalty area', 'Is_Home',
]
categorical_features = ['Formation_Clean', 'Season_Phase', 'Possession_Tier']

for col in categorical_features:
    df[col] = df[col].astype(str)

X = df[numeric_features + categorical_features]
# target: points earned per match (0 = loss, 1 = draw, 3 = win)
y = df['Points']

print(f'Features : {len(numeric_features)} numeric + {len(categorical_features)} categorical')
print(f'Samples  : {len(X)}')

Features : 17 numeric + 3 categorical
Samples  : 716


## Model

In [6]:
# scale numerics to zero mean and encode categoricals as dummy variables
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

ridge_pipe = Pipeline([('pre', preprocessor), ('model', Ridge())])
ridge_gs = GridSearchCV(
    ridge_pipe,
    {'model__alpha': [0.01, 0.1, 1, 10, 100]},
    cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1
)
ridge_gs.fit(X, y)

print(f'Ridge best alpha : {ridge_gs.best_params_["model__alpha"]}')
print(f'Ridge CV RMSE    : {-ridge_gs.best_score_:.4f}')

Ridge best alpha : 100
Ridge CV RMSE    : 0.9247


In [7]:
gbt_pipe = Pipeline([('pre', preprocessor), ('model', HistGradientBoostingRegressor(random_state=42))])
gbt_gs = GridSearchCV(
    gbt_pipe,
    {
        'model__max_depth':     [2, 3, 4],
        'model__learning_rate': [0.05, 0.1],
        'model__max_iter':      [100, 200],
    },
    cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1
)
gbt_gs.fit(X, y)

print(f'GBT best params : {gbt_gs.best_params_}')
print(f'GBT CV RMSE     : {-gbt_gs.best_score_:.4f}')

GBT best params : {'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__max_iter': 200}
GBT CV RMSE     : 0.9388


## Model Performance

In [8]:
# CV RMSE is the out-of-sample estimate; Train RMSE shows how well the model fit the data it was trained on
def evaluate(gs, name):
    best  = gs.best_estimator_
    y_hat = best.predict(X)
    rmse  = mean_squared_error(y, y_hat) ** 0.5
    mae   = mean_absolute_error(y, y_hat)
    r2    = r2_score(y, y_hat)
    print(f'{name}')
    print(f'  CV RMSE (5-fold) : {-gs.best_score_:.4f}')
    print(f'  Train RMSE       : {rmse:.4f}')
    print(f'  Train MAE        : {mae:.4f}')
    print(f'  Train R²         : {r2:.4f}')
    print()

evaluate(ridge_gs, 'Ridge Regression')
evaluate(gbt_gs,   'Gradient Boosted Trees')

Ridge Regression
  CV RMSE (5-fold) : 0.9247
  Train RMSE       : 0.8924
  Train MAE        : 0.7493
  Train R²         : 0.5320

Gradient Boosted Trees
  CV RMSE (5-fold) : 0.9388
  Train RMSE       : 0.7681
  Train MAE        : 0.6302
  Train R²         : 0.6533



In [9]:
# larger absolute coefficient = stronger influence on predicted points
ridge_best  = ridge_gs.best_estimator_
cat_encoded = (
    ridge_best.named_steps['pre']
    .named_transformers_['cat']
    .get_feature_names_out(categorical_features)
    .tolist()
)
all_names = numeric_features + cat_encoded

coef_df = (
    pd.DataFrame({'Feature': all_names, 'Coefficient': ridge_best.named_steps['model'].coef_})
    .assign(Abs=lambda d: d['Coefficient'].abs())
    .sort_values('Abs', ascending=False)
    .drop(columns='Abs')
    .reset_index(drop=True)
)

print('Ridge Coefficients (top 15):')
print(coef_df.head(15).to_string(index=False))

Ridge Coefficients (top 15):
                Feature  Coefficient
                xG_Diff     0.428768
Shots against on target    -0.372245
                     xG     0.271756
   Shot_Conversion_Rate     0.196241
        Shots on target     0.195512
           Possession %    -0.157232
   Possession_Tier_High    -0.108168
Touches in penalty area    -0.107717
    Possession_Tier_Low     0.096655
                Is_Home     0.089192
  Formation_Clean_5-3-2    -0.085637
            GK_Claiming     0.084083
Formation_Clean_3-4-1-2    -0.076161
                   PPDA    -0.074900
        GK_Shotstopping     0.068646


In [10]:
# permutation importance measures how much accuracy drops when a feature's values are randomly shuffled
perm = permutation_importance(
    gbt_gs.best_estimator_, X, y,
    n_repeats=10, random_state=42, n_jobs=-1
)

imp_df = (
    pd.DataFrame({
        'Feature':    numeric_features + categorical_features,
        'Importance': perm.importances_mean,
    })
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)

print('GBT Permutation Importance (top 15):')
print(imp_df.head(15).to_string(index=False))

GBT Permutation Importance (top 15):
                Feature  Importance
Shots against on target    0.198778
                xG_Diff    0.176517
   Shot_Conversion_Rate    0.173723
           Possession %    0.087557
                     xG    0.064303
        Shots on target    0.036233
        GK_Shotstopping    0.024447
            GK_Claiming    0.018876
                   PPDA    0.014619
Touches in penalty area    0.011366
                Is_Home    0.009344
             GK_Passing    0.007920
  Deep completed passes    0.004865
        Formation_Clean    0.003896
            GK_Fielding    0.001293
